# 03 · Model development — beating $116 per contact (Rodolfo)

**What this notebook does.** Builds and validates the model for measurement 1 — Albert's ordering, item (1): *the ranking result at stewardship capacity against the ladder of baselines*. All logic lives in `src/model.py`; this notebook runs it, shows the evidence, and records the decisions, so nothing here is copy-pasted logic (repo rule: shared code in `src/`).

**The bar.** Rank-by-first-gift-size finds **$116 per contact** at 10% capacity on citizen donors. The reference logistic on donations-file features alone got $119 — a rounding error. Its lesson (see README): *the signal is not in the donations file*; it is in Reid's projects join, and specifically in interactions (state × gift decile, subject × gift decile, grade × gift decile).

**The design, in three sentences.** A gradient-boosted classifier predicts P(second gift within 12 months); a gradient-boosted regressor on log(amount), fit on returners only, predicts the size of that gift (Albert: "a regression on the log of the amount" is the approved upgrade over the cohort median). We rank donors two ways — by probability, and by probability × expected amount — because the headline metric is *dollars* per contact, and first-gift size predicts the size of the second gift more than the fact of it. Trees learn Reid's interactions from the marginal columns directly, without us hand-picking thousands of cross terms to overfit.

**What is deliberately excluded.** `THANK_YOU_PACKET_MAILED` (no timing in the codebook — possible leakage, Albert pre-approved dropping), `TEACHER_ID` (memorization), any post-outcome project field.

**🔴 The holdout is not touched anywhere in this notebook.** Development uses only the train split, re-split by time: fit on cohorts through 2015, validate on the 2016 cohorts — the last pre-holdout year, chosen because behaviour drifts (reference coefficient −0.21 on cohort year) and a model that transfers 2015→2016 is our best evidence it will transfer 2016→2017/18. Random K-fold would let the model see the future and overstate everything.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
import config
import model as M

# Reid's notebook-02 output: cohorts + first-project features. This notebook refuses to run on the
# plain cohorts file, because a donations-only "result" would repeat the reference model's mistake.
COHORTS = REPO / "data" / "processed" / "cohorts_with_projects.parquet"
assert COHORTS.exists(), "Run notebooks/02 first — the model needs the projects join."
print("Using", COHORTS)

## 1 · The development run

One call. Internally: population filter from `config` (citizen donors — the scoping decision the team confirmed), fit on cohorts ≤ 2015-12, evaluate on 2016 cohorts through the **same scorer and same baseline ladder as every other number in the repo** (`evals/score.py`, `src/baselines.py`), capacity ranking within cohort month.

What to read in the output, in order: (1) the table at 10% capacity — `model_expected_value` vs `gift_amount` on **value_per_contact** is the whole contest; (2) the bootstrap noise floor — AGENTS.md requires the standard error next to any leaderboard number, and a gain smaller than ~2× the combined SE is not a gain; (3) the ROC-AUC line is context only — Albert was explicit that the headline is the ranking result at capacity, not an AUC.

In [ ]:
dev_table = M.run(str(COHORTS))          # development only — never touches the holdout
dev_table[dev_table.capacity == config.STEWARDSHIP_CAPACITY]

## 2 · The capacity sweep — the result is not one cherry-picked operating point

`config.CAPACITY_SWEEP` reports the same comparison at 1/5/10/20/50%. If the model only wins at exactly 10%, that is fragility worth reporting; if the curve is above the baseline throughout, the recommendation survives Malorie's cost sensitivity (her break-even probability moves the effective capacity).

In [ ]:
pivot = dev_table.pivot_table(index="capacity", columns="ranking", values="value_per_contact")
cols = [c for c in ["model_expected_value", "model_p_return", "gift_amount",
                    "first_month_gift_count", "random"] if c in pivot.columns]
pivot[cols].round(1)

## 3 · What the model uses — permutation importance on dev-val

Not for tuning — for the presentation. The claim "the signal lives in the projects join" needs a figure: if state/subject/grade and their implicit crosses with gift size do not surface here, the claim dies in this cell instead of on slide 7. Computed on a 50k sample of dev-val for speed; importance = drop in ROC-AUC when a column is shuffled.

In [ ]:
import numpy as np
from sklearn.inspection import permutation_importance

cohorts = pd.read_parquet(COHORTS)
pop = config.STAKEHOLDER_POPULATION
if pop and "donor_type" in cohorts.columns:
    cohorts = cohorts[cohorts["donor_type"] == pop]
train = cohorts[cohorts["split"] == "train"].reset_index(drop=True)
cut = M.month_str_to_index(config.TRAIN_COHORT_END) - 11
dev_train = train[train["cohort_month"] < cut].reset_index(drop=True)
dev_val   = train[train["cohort_month"] >= cut].reset_index(drop=True)

clf, reg, predict, have_projects, feat_cols = M.fit_models(dev_train)
sample = dev_val.sample(n=min(50_000, len(dev_val)), random_state=M.RANDOM_STATE)

X, cat_mask, _ = M.design_matrix(sample)
from sklearn.preprocessing import OrdinalEncoder
Xt, _, _ = M.design_matrix(dev_train)
cat_cols = list(Xt.columns[cat_mask])
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan,
                     encoded_missing_value=np.nan).fit(Xt[cat_cols])
X[cat_cols] = enc.transform(X[cat_cols])  # design_matrix already normalizes pd.NA -> np.nan
imp = permutation_importance(clf, X.astype(float).values, sample["gave_again"].values,
                             n_repeats=5, random_state=M.RANDOM_STATE, scoring="roc_auc")
pd.Series(imp.importances_mean, index=X.columns).sort_values(ascending=False).round(4)

## 4 · Error analysis hand-off, and what we are *not* claiming

Two things Thadeus needs from this run: dev-val predictions carry the drift question (does precision decay 2015→2016 the way the −0.21 cohort coefficient predicts?), and the probability outputs feed his calibration curve — nothing in our fitting optimized calibration, so his measurement 2 is a genuine check, not a formality.

**Not claimed:** any uplift from being contacted. There is no randomized outreach in this data; we rank by predicted future value and report value *identified*. The presentation says this out loud (it is written into `evals/score.py`'s docstring for the same reason).

## 5 · 🔴 The holdout gate

The cell below is intentionally not runnable as-is. Scoring the holdout happens **once**, from the command line, after the team has seen the dev-val table above and agreed in the chat that the model is final. It refits on the full train split and writes `data/processed/model_scores.parquet`, which replaces `reference_scores.parquet` as the input to Malorie's decision layer (`src/decision.py`) and Thadeus's calibration.

```
# AFTER team sign-off in the chat — run exactly once:
# python src/model.py data/processed/cohorts_with_projects.parquet --holdout
```

If the holdout number then lands below the dev-val number, that gap is the drift story, and it goes on a slide — a discrepancy is a finding, not a bug to hide.